In [1]:
!pip install -q pandas numpy scikit-learn tensorflow nltk transformers datasets accelerate torch

In [2]:
import os
import re
import random
import inspect
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("TensorFlow version:", tf.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

TensorFlow version: 2.20.0
PyTorch version: 2.10.0+cu128
CUDA available: True


In [4]:
from google.colab import files
uploaded = files.upload()

Saving CEAS_08_cleaned.csv to CEAS_08_cleaned.csv
Saving machinewars_filtered_emails.json to machinewars_filtered_emails.json
Saving Nazario_cleaned.csv to Nazario_cleaned.csv
Saving Nigerian_Fraud_cleaned.csv to Nigerian_Fraud_cleaned.csv
Saving SpamAssasin_cleaned.csv to SpamAssasin_cleaned.csv


In [6]:
import pandas as pd
import json
from pathlib import Path


# ---------------------------------
# 1. Helpers
# ---------------------------------
def safe_str(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def build_text_subject_body(row):
    subject = safe_str(row.get("subject", ""))
    body = safe_str(row.get("body", ""))
    return f"{subject}\n\n{body}".strip()


# ---------------------------------
# 2. Label handling
# ---------------------------------
def normalize_machinewars_label(label, spam_as_phishing=False):
    """
    MachineWars:
      Phishing -> 1
      Legitimate/Valid/Ham -> 0
      Spam -> 1 if spam_as_phishing=True else excluded
    """
    if pd.isna(label):
        return None

    label = str(label).strip().lower()

    if label == "phishing":
        return 1

    if label in {"legitimate", "valid", "ham", "benign", "safe"}:
        return 0

    if label == "spam":
        return 1 if spam_as_phishing else None

    return None


def normalize_test_label(label):
    """
    For CEAS-style test sets.
    Spam is excluded here unless you explicitly want otherwise.
    """
    if pd.isna(label):
        return None

    label = str(label).strip().lower()

    if label == "phishing":
        return 1

    if label in {"legitimate", "valid", "ham", "benign", "safe"}:
        return 0

    return None


# ---------------------------------
# 3. MachineWars loader
# ---------------------------------
def load_machinewars(json_path_or_list, spam_as_phishing=False, dataset_name="machinewars"):
    """
    Expected MachineWars fields:
      sender, subject, body, type, url
    """
    if isinstance(json_path_or_list, (str, Path)):
        with open(json_path_or_list, "r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        data = json_path_or_list

    df = pd.DataFrame(data).copy()

    # rename type -> label
    if "type" in df.columns:
        df = df.rename(columns={"type": "label"})

    # ensure required columns exist
    for col in ["subject", "body", "label"]:
        if col not in df.columns:
            df[col] = ""

    # keep optional columns if they exist
    if "sender" not in df.columns:
        df["sender"] = ""

    if "url" not in df.columns:
        df["url"] = ""

    # flatten url list to string
    df["url"] = df["url"].apply(
        lambda x: " | ".join(x) if isinstance(x, list) else safe_str(x)
    )

    df["dataset"] = dataset_name
    df["label_raw"] = df["label"].astype(str).str.strip().str.lower()
    df["label_id"] = df["label_raw"].apply(
        lambda x: normalize_machinewars_label(x, spam_as_phishing=spam_as_phishing)
    )

    # drop excluded rows
    df = df[df["label_id"].notna()].copy()
    df["label_id"] = df["label_id"].astype(int)

    # normalized binary label name
    df["label"] = df["label_id"].map({0: "legitimate", 1: "phishing"})

    # only subject + body for model text
    df["text"] = df.apply(build_text_subject_body, axis=1)

    # final schema
    df = df[
        ["dataset", "sender", "subject", "body", "url", "label_raw", "label", "label_id", "text"]
    ]

    return df


# ---------------------------------
# 4. CEAS-style test loader
# ---------------------------------
def load_ceas_style_csv(csv_path, dataset_name=None):
    """
    Assumes CEAS-style columns similar to:
      subject, body, label
    """
    csv_path = Path(csv_path)
    if dataset_name is None:
        dataset_name = csv_path.stem

    df = pd.read_csv(csv_path).copy()

    rename_map = {
        "Subject": "subject",
        "Body": "body",
        "Label": "label",
        "type": "label",
    }
    df = df.rename(columns=rename_map)

    for col in ["subject", "body", "label"]:
        if col not in df.columns:
            df[col] = ""

    if "sender" not in df.columns:
        df["sender"] = ""

    if "url" not in df.columns:
        df["url"] = ""

    df["dataset"] = dataset_name
    df["label_raw"] = df["label"].astype(str).str.strip().str.lower()
    df["label_id"] = df["label_raw"].apply(normalize_test_label)

    # drop non-binary rows
    df = df[df["label_id"].notna()].copy()
    df["label_id"] = df["label_id"].astype(int)
    df["label"] = df["label_id"].map({0: "legitimate", 1: "phishing"})
    df["text"] = df.apply(build_text_subject_body, axis=1)

    df = df[
        ["dataset", "sender", "subject", "body", "url", "label_raw", "label", "label_id", "text"]
    ]

    return df


# ---------------------------------
# 5. Create the two MachineWars versions
# ---------------------------------
machinewars_path = "machinewars_filtered_emails.json"

# Version A: spam merged into phishing
machinewars_spam_as_phishing_df = load_machinewars(
    machinewars_path,
    spam_as_phishing=True,
    dataset_name="machinewars"
)

# Version B: spam removed
machinewars_no_spam_df = load_machinewars(
    machinewars_path,
    spam_as_phishing=False,
    dataset_name="machinewars"
)

print("MachineWars: spam merged into phishing")
print(machinewars_spam_as_phishing_df["label"].value_counts())
print(machinewars_spam_as_phishing_df.head(3))

print("\nMachineWars: spam removed")
print(machinewars_no_spam_df["label"].value_counts())
print(machinewars_no_spam_df.head(3))


# ---------------------------------
# 6. Load the 4 CEAS-style test datasets
# ---------------------------------
test_paths = [
    "CEAS_08_cleaned.csv",
    "Nazario_cleaned.csv",
    "Nigerian_Fraud_cleaned.csv",
    "SpamAssasin_cleaned.csv",
]

test_dfs = [load_ceas_style_csv(p) for p in test_paths]

for i, df in enumerate(test_dfs, 1):
    print(f"\nTest dataset {i}:")
    print(df["label"].value_counts())
    print(df.head(2))

MachineWars: spam merged into phishing
label
phishing      13200
legitimate     6600
Name: count, dtype: int64
       dataset                                             sender  \
0  machinewars      Dropbox Security <noreply@dropbox-secure.net>   
1  machinewars  Google Drive Security <security-alert@google-d...   
2  machinewars  Microsoft OneDrive Security <noreply@microsoft...   

                                             subject  \
0  Unusual Sign-in Activity Detected on Your Drop...   
1  Security Alert: New Sign-in to Your Google Dri...   
2  Important Security Notification Regarding Your...   

                                                body  \
0  Dear User,\n\nWe've detected an unusual sign-i...   
1  Google Drive Security Alert\n\nWe've noticed a...   
2  Hello sarah.smith@gmail.com,\n\nThis is an aut...   

                                                 url label_raw     label  \
0         https://dropbox-security.co/account/review  phishing  phishing   
1  https:/

In [7]:
train_df, val_df = train_test_split(
    machinewars_no_spam_df,
    test_size=0.2,
    random_state=SEED,
    stratify=machinewars_no_spam_df["label_id"]
)

In [8]:
from datasets import Dataset

train_ds = Dataset.from_pandas(
    train_df[["text", "label_id"]]
    .rename(columns={"label_id": "labels"})
    .reset_index(drop=True)
)

val_ds = Dataset.from_pandas(
    val_df[["text", "label_id"]]
    .rename(columns={"label_id": "labels"})
    .reset_index(drop=True)
)

In [9]:
def compute_binary_metrics(y_true, y_pred, y_prob):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except Exception:
        roc_auc = float("nan")

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
    }
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # If model returns tuple-like predictions, keep logits only.
    if isinstance(logits, tuple):
        logits = logits[0]

    y_true = labels
    y_pred = np.argmax(logits, axis=-1)

    # Convert logits to probability for class 1.
    exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    y_prob = probs[:, 1]

    return compute_binary_metrics(y_true, y_pred, y_prob)

In [10]:
import gc

X_train = train_df["text"].astype(str).tolist()
y_train = train_df["label_id"].astype(int).values

X_val = val_df["text"].astype(str).tolist()
y_val = val_df["label_id"].astype(int).values

DISTILBERT_MODEL_NAME = "microsoft/deberta-v3-base"
DISTILBERT_MAX_LENGTH = 512

id2label = {0: "legitimate", 1: "phishing"}
label2id = {"legitimate": 0, "phishing": 1}

gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"

distilbert_tokenizer = AutoTokenizer.from_pretrained(DISTILBERT_MODEL_NAME)

distilbert_model = AutoModelForSequenceClassification.from_pretrained(
    DISTILBERT_MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

distilbert_model = distilbert_model.to(device=device, dtype=torch.float32)

print("Model parameter dtype:", next(distilbert_model.parameters()).dtype)
print("Model device:", next(distilbert_model.parameters()).device)


def tokenize_distilbert_batch(batch):
    return distilbert_tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=DISTILBERT_MAX_LENGTH,
    )


train_dataset = train_ds.map(tokenize_distilbert_batch, batched=True)
eval_dataset = val_ds.map(tokenize_distilbert_batch, batched=True)

keep_cols = {"input_ids", "attention_mask", "labels"}
train_dataset = train_dataset.remove_columns([c for c in train_dataset.column_names if c not in keep_cols])
eval_dataset = eval_dataset.remove_columns([c for c in eval_dataset.column_names if c not in keep_cols])

data_collator = DataCollatorWithPadding(tokenizer=distilbert_tokenizer)


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias          

Model parameter dtype: torch.float32
Model device: cuda:0


Map:   0%|          | 0/10684 [00:00<?, ? examples/s]

Map:   0%|          | 0/2672 [00:00<?, ? examples/s]

In [25]:
DISTILBERT_BATCH_SIZE = 16
DISTILBERT_EPOCHS = 3


def compute_distilbert_trainer_metrics(eval_pred):
    logits, labels = eval_pred

    if isinstance(logits, tuple):
        logits = logits[0]

    logits = np.asarray(logits, dtype=np.float32)

    if not np.isfinite(logits).all():
        print("WARNING: non-finite logits detected in metrics")
        print("NaN count:", np.isnan(logits).sum())
        print("Inf count:", np.isinf(logits).sum())
        logits = np.nan_to_num(logits, nan=0.0, posinf=1e4, neginf=-1e4)

    probs_all = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()
    probs = probs_all[:, 1]
    preds = (probs >= 0.5).astype(int)
    return compute_binary_metrics(labels, preds, probs)


class DebugTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").long()
        outputs = model(**inputs)
        logits = outputs.logits.float()

        if not torch.isfinite(logits).all():
            print("Bad logits:")
            print(logits)
            raise ValueError("Non-finite logits during training")

        loss = torch.nn.functional.cross_entropy(logits, labels)

        if not torch.isfinite(loss):
            print("Bad loss:", loss)
            print("Logits:", logits)
            print("Labels:", labels)
            raise ValueError("Non-finite loss during training")

        return (loss, outputs) if return_outputs else loss


print("Train labels:")
print(train_df["label_id"].value_counts(normalize=False))
print(train_df["label_id"].value_counts(normalize=True))

print("Validation labels:")
print(val_df["label_id"].value_counts(normalize=False))
print(val_df["label_id"].value_counts(normalize=True))

print("Empty train texts:", (train_df["text"].astype(str).str.strip() == "").sum())
print("Empty val texts:", (val_df["text"].astype(str).str.strip() == "").sum())

debug_batch = next(iter(torch.utils.data.DataLoader(
    train_dataset,
    batch_size=4,
    collate_fn=data_collator,
)))

print("Debug batch keys:", debug_batch.keys())
print("input_ids shape:", debug_batch["input_ids"].shape)
print("attention_mask shape:", debug_batch["attention_mask"].shape)
print("labels:", debug_batch["labels"])
print("first input_ids:", debug_batch["input_ids"][0][:30])
print("first attention_mask:", debug_batch["attention_mask"][0][:30])

debug_batch = {k: v.to(device) for k, v in debug_batch.items()}

distilbert_model.eval()
with torch.no_grad():
    if device == "cuda":
        with torch.autocast(device_type="cuda", enabled=False):
            outputs = distilbert_model(
                input_ids=debug_batch["input_ids"],
                attention_mask=debug_batch["attention_mask"],
            )
    else:
        outputs = distilbert_model(
            input_ids=debug_batch["input_ids"],
            attention_mask=debug_batch["attention_mask"],
        )

print("Pre-training logits:")
print(outputs.logits)
print("Finite logits:", torch.isfinite(outputs.logits).all())
print("Logits dtype:", outputs.logits.dtype)
print("Labels dtype:", debug_batch["labels"].dtype)

assert torch.isfinite(outputs.logits).all(), "Model produces NaN/Inf logits before training."
assert outputs.logits.dtype == torch.float32, "Model is not producing fp32 logits."


# Transformers renamed evaluation_strategy to eval_strategy in newer releases.
import inspect
training_args_kwargs = dict(
    output_dir="/content/deberta_subject_body_model",
    learning_rate=1e-5,
    per_device_train_batch_size=DISTILBERT_BATCH_SIZE,
    per_device_eval_batch_size=DISTILBERT_BATCH_SIZE,
    num_train_epochs=DISTILBERT_EPOCHS,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    seed=SEED,
    fp16=False,
    bf16=False,
    max_grad_norm=1.0,
)
if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
    training_args_kwargs["eval_strategy"] = "epoch"
else:
    training_args_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_args_kwargs)

print("Before training parameter sample:")
before_params = next(distilbert_model.parameters()).detach().flatten()[:5].clone()
print(before_params)

distilbert_trainer = DebugTrainer(
    model=distilbert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_distilbert_trainer_metrics,
)

train_result = distilbert_trainer.train()

print("After training parameter sample:")
after_params = next(distilbert_model.parameters()).detach().flatten()[:5].clone()
print(after_params)
print("Parameters changed:", not torch.equal(before_params.cpu(), after_params.cpu()))
print("Train result:")
print(train_result)
print("DeBERTa training complete.")


Train labels:
label_id
1    5404
0    5280
Name: count, dtype: int64
label_id
1    0.505803
0    0.494197
Name: proportion, dtype: float64
Validation labels:
label_id
1    1352
0    1320
Name: count, dtype: int64
label_id
1    0.505988
0    0.494012
Name: proportion, dtype: float64
Empty train texts: 0
Empty val texts: 0
Debug batch keys: KeysView({'labels': tensor([0, 1, 0, 1]), 'input_ids': tensor([[34359,   294, 13643,  ...,     0,     0,     0],
        [42907,   294, 17509,  ...,     0,     0,     0],
        [  730, 25892,   294,  ...,  4198, 12478,   260],
        [24870,   294,   730,  ...,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]])})
input_ids shape: torch.Size([4, 451])
attention_mask shape: torch.Size([4, 451])
labels: tensor([0, 1, 0, 1])
first input_ids: tensor([34359,   294, 13643, 30259,   510,  5540,   270, 32731,  8510,  5365,
        

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.116994,0.102312,0.971557,0.994574,0.948964,0.971234,0.998673
2,0.001004,0.030979,0.993638,0.996283,0.991124,0.993697,0.999666
3,0.005419,0.028844,0.994012,0.996285,0.991864,0.994070,0.999812


After training parameter sample:
tensor([-0.0014, -0.0116,  0.0114,  0.0068, -0.0196], device='cuda:0')
Parameters changed: True
Train result:
TrainOutput(global_step=2004, training_loss=0.07841101247335583, metrics={'train_runtime': 1056.265, 'train_samples_per_second': 30.345, 'train_steps_per_second': 1.897, 'total_flos': 8406326973832128.0, 'train_loss': 0.07841101247335583, 'epoch': 3.0})
DeBERTa training complete.


In [11]:
def _model_safe_text(text):
    """
    DeBERTa can crash if an attack produces an empty/whitespace-only string.
    Keep inference inputs non-empty while preserving normal text unchanged.
    """
    s = "" if text is None else str(text)
    s = s.strip()
    return s if s else "[EMPTY_EMAIL]"


def distilbert_predict_one(text):
    pred, prob = distilbert_batch_predict([text], batch_size=1)
    return int(pred[0]), float(prob[0])


def distilbert_batch_predict(texts, batch_size=32):
    texts = [_model_safe_text(t) for t in texts]
    if len(texts) == 0:
        return np.array([], dtype=int), np.array([], dtype=float)

    distilbert_model.eval()
    all_probs = []
    device = next(distilbert_model.parameters()).device

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        encoded = distilbert_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=DISTILBERT_MAX_LENGTH,
            return_tensors="pt",
            add_special_tokens=True,
        )

        # Defensive check: this catches empty tokenization before DeBERTa crashes.
        if encoded["input_ids"].shape[1] == 0:
            batch_texts = ["[EMPTY_EMAIL]" for _ in batch_texts]
            encoded = distilbert_tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=DISTILBERT_MAX_LENGTH,
                return_tensors="pt",
                add_special_tokens=True,
            )

        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            if device.type == "cuda":
                with torch.autocast(device_type="cuda", enabled=False):
                    logits = distilbert_model(**encoded).logits.float()
            else:
                logits = distilbert_model(**encoded).logits.float()

            if not torch.isfinite(logits).all():
                raise ValueError("Non-finite logits during prediction")

            probs = torch.softmax(logits, dim=-1)[:, 1]

        all_probs.append(probs.detach().cpu().numpy())

    probs = np.concatenate(all_probs)
    preds = (probs >= 0.5).astype(int)
    return preds, probs


def evaluate_distilbert(df_eval):
    y_true = df_eval["label_id"].astype(int).values
    y_pred, y_prob = distilbert_batch_predict(df_eval["text"].astype(str).tolist())
    return compute_binary_metrics(y_true, y_pred, y_prob)



In [11]:
print("Validation metrics:")
print(evaluate_distilbert(val_df))

distilbert_rows = [{"dataset": "validation", **evaluate_distilbert(val_df)}]

for test_df in test_dfs:
    test_name = test_df["dataset"].iloc[0]
    metrics = evaluate_distilbert(test_df)
    distilbert_rows.append({"dataset": test_name, **metrics})

distilbert_results_df = pd.DataFrame(distilbert_rows)
distilbert_results_df

Validation metrics:
{'accuracy': 0.9932634730538922, 'precision': 0.9962797619047619, 'recall': 0.9903846153846154, 'f1': 0.9933234421364985, 'roc_auc': np.float64(0.999743925945849)}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,dataset,accuracy,precision,recall,f1,roc_auc
0,validation,0.993263,0.996280,0.990385,0.993323,0.999744
1,CEAS_08_cleaned,0.606579,0.976748,0.301941,0.461286,0.946572
2,Nazario_cleaned,0.800639,1.000000,0.800639,0.889283,NaN
3,Nigerian_Fraud_cleaned,0.827431,1.000000,0.827431,0.905567,NaN
4,SpamAssasin_cleaned,0.792391,0.984848,0.302678,0.463045,0.946274


Validation metrics:
{'accuracy': 0.9651515151515152, 'precision': 0.9742228961334344, 'recall': 0.9734848484848485, 'f1': 0.9738537324744221, 'roc_auc': np.float64(0.9911992079889806)}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,dataset,accuracy,precision,recall,f1,roc_auc
0,validation,0.965152,0.974223,0.973485,0.973854,0.991199
1,CEAS_08_cleaned,0.862517,0.931293,0.813570,0.868460,0.926345
2,Nazario_cleaned,0.941853,1.000000,0.941853,0.970056,NaN
3,Nigerian_Fraud_cleaned,0.934574,1.000000,0.934574,0.966181,NaN
4,SpamAssasin_cleaned,0.811672,0.694757,0.647846,0.670482,0.866153


In [12]:
import nltk
from nltk.corpus import wordnet

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [13]:
def benign_prefix_attack(text):
    prefix = (
        "Hello,\n\n"
        "This is a routine business communication regarding normal account activity. "
        "Please review the information below.\n\n"
    )
    return prefix + str(text)


def benign_suffix_attack(text):
    suffix = (
        "\n\nThank you for your attention."
        "\nThis message is part of a standard informational workflow."
        "\nBest regards."
    )
    return str(text) + suffix


def contradiction_attack(text):
    prefix = (
        "Important note: this is a legitimate and safe message. "
        "It does not request any unsafe action.\n\n"
    )
    return prefix + str(text)


def prefix_injection_attack(text):
    prefix = (
        "This is a normal and trustworthy business email. "
        "The message is legitimate, safe, and routine.\n\n"
    )
    return prefix + str(text)


PHISHING_KEYWORDS = {
    "verify", "verification", "account", "password", "login", "signin",
    "security", "alert", "urgent", "confirm", "suspend", "suspended",
    "click", "update", "reset", "limited", "immediately"
}

def keyword_deletion_attack(text, max_delete=5):
    words = str(text).split()
    new_words = []
    deleted = 0

    for w in words:
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w).lower()
        if clean in PHISHING_KEYWORDS and deleted < max_delete:
            deleted += 1
            continue
        new_words.append(w)

    return " ".join(new_words)


def get_synonyms(word):
    syns = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            s = lemma.name().replace("_", " ").strip()
            if s and s.lower() != word.lower():
                syns.add(s)
    return list(syns)


def synonym_attack(text, replace_prob=0.12, max_replacements=8, seed=42):
    rng = random.Random(seed)
    words = str(text).split()
    new_words = []
    replacements = 0

    for w in words:
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w)

        if (
            replacements < max_replacements
            and len(clean) >= 4
            and clean.isalpha()
            and rng.random() < replace_prob
        ):
            syns = get_synonyms(clean)
            syns = [s for s in syns if s.isalpha() and len(s.split()) == 1]

            if syns:
                replacement = rng.choice(syns)
                if w.istitle():
                    replacement = replacement.title()
                new_words.append(replacement)
                replacements += 1
                continue

        new_words.append(w)

    return " ".join(new_words)

In [14]:
predict_one = distilbert_predict_one
batch_predict = distilbert_batch_predict
ACTIVE_MODEL_NAME = "deberta"
print("Active model:", ACTIVE_MODEL_NAME)


Active model: deberta


In [15]:
def evaluate_attack_common(df_eval, attack_name, attack_fn):
    attacked_texts = [attack_fn(t) for t in df_eval["text"].astype(str).tolist()]
    y_true = df_eval["label_id"].astype(int).values

    y_pred, y_prob = batch_predict(attacked_texts)

    return {
        "attack": attack_name,
        "n_samples": len(df_eval),
        **compute_binary_metrics(y_true, y_pred, y_prob)
    }

In [16]:
attack_rows_val = []

attack_rows_val.append(evaluate_attack_common(val_df, "clean", lambda x: x))
attack_rows_val.append(evaluate_attack_common(val_df, "benign_prefix", benign_prefix_attack))
attack_rows_val.append(evaluate_attack_common(val_df, "benign_suffix", benign_suffix_attack))
attack_rows_val.append(evaluate_attack_common(val_df, "contradiction", contradiction_attack))
attack_rows_val.append(
    evaluate_attack_common(
        val_df,
        "synonym_attack",
        lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42)
    )
)
attack_rows_val.append(
    evaluate_attack_common(
        val_df,
        "keyword_deletion",
        lambda x: keyword_deletion_attack(x, max_delete=5)
    )
)
attack_rows_val.append(evaluate_attack_common(val_df, "prefix_injection", prefix_injection_attack))

attack_results_val_df = pd.DataFrame(attack_rows_val).sort_values("f1", ascending=False)
attack_results_val_df

,attack,n_samples,accuracy,precision,recall,f1,roc_auc
4,synonym_attack,2672,0.994386,0.994819,0.994083,0.994451,0.999720
2,benign_suffix,2672,0.994012,0.994815,0.993343,0.994078,0.999732
5,keyword_deletion,2672,0.994012,0.997765,0.990385,0.994061,0.999704
0,clean,2672,0.993263,0.996280,0.990385,0.993323,0.999744
3,contradiction,2672,0.992515,0.993333,0.991864,0.992598,0.999660
1,benign_prefix,2672,0.986901,0.976828,0.997781,0.987194,0.999638
6,prefix_injection,2672,0.985778,0.974026,0.998521,0.986121,0.999658


In [17]:
all_attack_tables = {}

for test_df in test_dfs:
    test_name = test_df["dataset"].iloc[0]

    rows = []
    rows.append(evaluate_attack_common(test_df, "clean", lambda x: x))
    rows.append(evaluate_attack_common(test_df, "benign_prefix", benign_prefix_attack))
    rows.append(evaluate_attack_common(test_df, "benign_suffix", benign_suffix_attack))
    rows.append(evaluate_attack_common(test_df, "contradiction", contradiction_attack))
    rows.append(
        evaluate_attack_common(
            test_df,
            "synonym_attack",
            lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42)
        )
    )
    rows.append(
        evaluate_attack_common(
            test_df,
            "keyword_deletion",
            lambda x: keyword_deletion_attack(x, max_delete=5)
        )
    )
    rows.append(evaluate_attack_common(test_df, "prefix_injection", prefix_injection_attack))

    result_df = pd.DataFrame(rows).sort_values("f1", ascending=False)
    all_attack_tables[test_name] = result_df

    print(f"\n=== {ACTIVE_MODEL_NAME} | {test_name} ===")
    print(result_df)


=== deberta | CEAS_08_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
1     benign_prefix      39154  0.832865   0.955298  0.734777  0.830651   
6  prefix_injection      39154  0.818333   0.955639  0.707170  0.812840   
2     benign_suffix      39154  0.704883   0.977798  0.481916  0.645628   
3     contradiction      39154  0.688333   0.977698  0.451607  0.617832   
4    synonym_attack      39154  0.618047   0.975031  0.323597  0.485923   
0             clean      39154  0.606579   0.976748  0.301941  0.461286   
5  keyword_deletion      39154  0.606171   0.978539  0.300613  0.459933   

    roc_auc  
1  0.935838  
6  0.914760  
2  0.956235  
3  0.945245  
4  0.945131  
0  0.946572  
5  0.947003  


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist


=== deberta | Nazario_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
6  prefix_injection       1565  0.913099        1.0  0.913099  0.954576   
1     benign_prefix       1565  0.897764        1.0  0.897764  0.946128   
3     contradiction       1565  0.838978        1.0  0.838978  0.912439   
4    synonym_attack       1565  0.831310        1.0  0.831310  0.907886   
2     benign_suffix       1565  0.825559        1.0  0.825559  0.904445   
0             clean       1565  0.800639        1.0  0.800639  0.889283   
5  keyword_deletion       1565  0.792971        1.0  0.792971  0.884533   

   roc_auc  
6      NaN  
1      NaN  
3      NaN  
4      NaN  
2      NaN  
0      NaN  
5      NaN  


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist


=== deberta | Nigerian_Fraud_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
1     benign_prefix       3332  0.886855        1.0  0.886855  0.940035   
6  prefix_injection       3332  0.883553        1.0  0.883553  0.938177   
4    synonym_attack       3332  0.836435        1.0  0.836435  0.910933   
3     contradiction       3332  0.831933        1.0  0.831933  0.908257   
2     benign_suffix       3332  0.831032        1.0  0.831032  0.907720   
0             clean       3332  0.827431        1.0  0.827431  0.905567   
5  keyword_deletion       3332  0.819328        1.0  0.819328  0.900693   

   roc_auc  
1      NaN  
6      NaN  
4      NaN  
3      NaN  
2      NaN  
0      NaN  
5      NaN  

=== deberta | SpamAssasin_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
1     benign_prefix       5809  0.823550   0.802620  0.534924  0.641984   
6  prefix_injection       5809  0.825960   0.844878  0.504075  0

In [ ]:
os.makedirs("/content/results", exist_ok=True)

distilbert_results_df.to_csv("/content/results/deberta_clean_results.csv", index=False)
attack_results_val_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_val_basic_attack_results.csv", index=False)

for dataset_name, df in all_attack_tables.items():
    safe_name = str(dataset_name).replace("/", "_").replace(" ", "_")
    df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_{safe_name}_basic_attack_results.csv", index=False)

print("Saved DeBERTa clean and basic attack results to /content/results")


In [ ]:
import shutil
shutil.make_archive("/content/phishing_deberta_results", "zip", "/content/results")

from google.colab import files
files.download("/content/phishing_deberta_results.zip")


In [16]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [17]:
def benign_prefix_attack(text):
    prefix = (
        "Hello,\n\n"
        "This is a routine business communication regarding normal account activity. "
        "Please review the information below.\n\n"
    )
    return prefix + str(text)


def benign_suffix_attack(text):
    suffix = (
        "\n\nThank you for your attention."
        "\nThis message is part of a standard informational workflow."
        "\nBest regards."
    )
    return str(text) + suffix


def contradiction_attack(text):
    prefix = (
        "Important note: this is a legitimate and safe message. "
        "It does not request any unsafe action.\n\n"
    )
    return prefix + str(text)


def training_context_attack(text):
    prefix = (
        "Example of a safe enterprise email used for routine staff communication:\n\n"
    )
    return prefix + str(text)


def noise_injection_attack(text):
    noise = (
        "\n\nReference ID: A7K2-91LM"
        "\nDepartment: operations"
        "\nCase Type: routine review"
    )
    return str(text) + noise

In [18]:
import re

STOPWORDS_ATTACK = {
    "the", "a", "an", "and", "or", "to", "of", "in", "for", "on", "at",
    "is", "are", "this", "that", "with", "from", "by", "as", "it", "be",
    "was", "were", "subject", "body", "dear", "hello", "hi", "regards",
    "thanks", "thank", "best"
}


def basic_tokenize_with_indices(text):
    """
    Splits text into whitespace-separated tokens and keeps positions.
    """
    tokens = str(text).split()
    return tokens


def is_deletable_token(tok):
    clean = re.sub(r"^[^\w]+|[^\w]+$", "", tok).lower()
    if len(clean) < 3:
        return False
    if clean in STOPWORDS_ATTACK:
        return False
    if not any(ch.isalpha() for ch in clean):
        return False
    return True


def delete_token_at_index(tokens, idx):
    text = " ".join(tokens[:idx] + tokens[idx+1:]).strip()
    return text if text else "[EMPTY_EMAIL]"


def greedy_delete_attack_blackbox(text, max_delete_steps=5, candidate_cap=15):
    """
    Black-box deletion attack:
    At each step, test candidate single-token deletions and keep the one that
    minimizes phishing probability.
    """
    current_text = str(text)

    for _ in range(max_delete_steps):
        tokens = basic_tokenize_with_indices(current_text)

        candidate_indices = [i for i, tok in enumerate(tokens) if is_deletable_token(tok)]

        if not candidate_indices:
            break

        # Cap candidates for speed
        candidate_indices = candidate_indices[:candidate_cap]

        candidate_texts = [_model_safe_text(delete_token_at_index(tokens, i)) for i in candidate_indices]

        # Candidate deletions can create empty strings for very short inputs; batch_predict is defensive too.
        _, candidate_probs = batch_predict(candidate_texts)
        best_idx = int(np.argmin(candidate_probs))

        best_text = candidate_texts[best_idx]

        if best_text == current_text:
            break

        current_text = best_text

    return current_text

def _model_safe_text(text):
    s = "" if text is None else str(text)
    s = s.strip()
    return s if s else "[EMPTY_EMAIL]"



In [19]:
def greedy_add_attack_blackbox(text, add_steps=3):
    """
    Greedily applies the benign addition that lowers phishing probability the most.
    """
    current_text = str(text)

    addition_fns = [
        ("benign_prefix", benign_prefix_attack),
        ("benign_suffix", benign_suffix_attack),
        ("contradiction", contradiction_attack),
        ("training_context", training_context_attack),
        ("noise_injection", noise_injection_attack),
    ]

    history = []

    for step in range(1, add_steps + 1):
        candidate_names = []
        candidate_texts = []

        for attack_name, attack_fn in addition_fns:
            candidate_names.append(attack_name)
            candidate_texts.append(attack_fn(current_text))

        candidate_preds, candidate_probs = batch_predict(candidate_texts)

        best_idx = int(np.argmin(candidate_probs))
        current_text = candidate_texts[best_idx]

        history.append({
            "step": step,
            "attack_name": candidate_names[best_idx],
            "pred": int(candidate_preds[best_idx]),
            "phishing_prob": float(candidate_probs[best_idx]),
        })

    return current_text, history

In [20]:
def add_only_attack(text, add_steps=3):
    attacked_text, _ = greedy_add_attack_blackbox(text, add_steps=add_steps)
    return attacked_text


def delete_only_attack(text, delete_steps=5):
    attacked_text = greedy_delete_attack_blackbox(text, max_delete_steps=delete_steps)
    return attacked_text


def hybrid_add_then_delete_attack_blackbox(text, add_steps=3, delete_steps=5):
    """
    First greedy additions, then greedy deletions.
    """
    current_text, add_history = greedy_add_attack_blackbox(text, add_steps=add_steps)
    current_text = greedy_delete_attack_blackbox(current_text, max_delete_steps=delete_steps)
    return current_text

In [21]:
def evaluate_attack_detailed(df_eval, attack_name, attack_fn, show_progress=True):
    y_true = df_eval["label_id"].astype(int).tolist()
    texts = df_eval["text"].astype(str).tolist()

    orig_pred, orig_prob = batch_predict(texts)

    attacked_texts = []
    iterator = texts
    if show_progress:
        iterator = tqdm(texts, total=len(texts), desc=f"{ACTIVE_MODEL_NAME} | {attack_name}")

    for text in iterator:
        attacked_texts.append(_model_safe_text(attack_fn(text)))

    new_pred, new_prob = batch_predict(attacked_texts)

    metrics = compute_binary_metrics(y_true, new_pred, new_prob)

    details_df = pd.DataFrame({
        "text": texts,
        "label_id": y_true,
        "orig_pred": orig_pred,
        "orig_prob": orig_prob,
        "attacked_text": attacked_texts,
        "new_pred": new_pred,
        "new_prob": new_prob,
    })

    details_df["flipped"] = details_df["orig_pred"] != details_df["new_pred"]
    details_df["prob_drop"] = details_df["orig_prob"] - details_df["new_prob"]

    summary = {
        "attack": attack_name,
        "n_samples": len(df_eval),
        "flip_rate": float(details_df["flipped"].mean()),
        "avg_prob_drop": float(details_df["prob_drop"].mean()),
        **metrics,
    }

    return summary, details_df

In [22]:
def evaluate_evasion_on_phishing(df_eval, attack_name, attack_fn, show_progress=True):
    """
    Restrict evaluation to:
    - true phishing samples
    - originally correct phishing predictions

    Reports attack success rate (ASR).
    """
    df_local = df_eval.copy()
    df_local = df_local[df_local["label_id"].astype(int) == 1].copy()

    if len(df_local) == 0:
        return {
            "attack": attack_name,
            "n_true_phishing": 0,
            "n_orig_correct_phishing": 0,
            "n_flipped": 0,
            "attack_success_rate": float("nan"),
            "robust_recall_on_orig_correct_phishing": float("nan"),
            "avg_prob_drop": float("nan"),
        }, pd.DataFrame()

    texts = df_local["text"].astype(str).tolist()
    orig_pred, orig_prob = batch_predict(texts)

    df_local["orig_pred"] = orig_pred
    df_local["orig_prob"] = orig_prob

    df_attack = df_local[df_local["orig_pred"] == 1].copy()

    if len(df_attack) == 0:
        return {
            "attack": attack_name,
            "n_true_phishing": len(df_local),
            "n_orig_correct_phishing": 0,
            "n_flipped": 0,
            "attack_success_rate": float("nan"),
            "robust_recall_on_orig_correct_phishing": float("nan"),
            "avg_prob_drop": float("nan"),
        }, pd.DataFrame()

    attacked_texts = []
    iterator = df_attack["text"].astype(str).tolist()
    if show_progress:
        iterator = tqdm(iterator, total=len(df_attack), desc=f"{ACTIVE_MODEL_NAME} | {attack_name} phishing")

    for text in iterator:
        attacked_texts.append(_model_safe_text(attack_fn(text)))

    new_pred, new_prob = batch_predict(attacked_texts)

    details_df = pd.DataFrame({
        "text": df_attack["text"].astype(str).tolist(),
        "label_id": df_attack["label_id"].astype(int).tolist(),
        "orig_pred": df_attack["orig_pred"].tolist(),
        "orig_prob": df_attack["orig_prob"].tolist(),
        "attacked_text": attacked_texts,
        "new_pred": new_pred,
        "new_prob": new_prob,
    })

    details_df["flipped"] = details_df["new_pred"] != 1
    details_df["prob_drop"] = details_df["orig_prob"] - details_df["new_prob"]

    n_orig_correct = len(details_df)
    n_flipped = int(details_df["flipped"].sum())
    asr = n_flipped / n_orig_correct
    robust_recall = 1.0 - asr

    summary = {
        "attack": attack_name,
        "n_true_phishing": len(df_local),
        "n_orig_correct_phishing": n_orig_correct,
        "n_flipped": n_flipped,
        "attack_success_rate": asr,
        "robust_recall_on_orig_correct_phishing": robust_recall,
        "avg_prob_drop": float(details_df["prob_drop"].mean()),
    }

    return summary, details_df

In [23]:
val_attack_summaries = []
val_attack_details = {}

attack_specs = [
    ("add_only_add3", lambda x: add_only_attack(x, add_steps=3)),
    ("delete_only_del5", lambda x: delete_only_attack(x, delete_steps=5)),
    ("hybrid_add3_delete5", lambda x: hybrid_add_then_delete_attack_blackbox(x, add_steps=3, delete_steps=5)),
]

In [26]:
for attack_name, attack_fn in attack_specs:
    summary, details = evaluate_attack_detailed(
        val_df,
        attack_name,
        attack_fn,
        show_progress=True
    )
    val_attack_summaries.append(summary)
    val_attack_details[attack_name] = details

val_attack_results_df = pd.DataFrame(val_attack_summaries).sort_values("f1", ascending=False)
val_attack_results_df

deberta | add_only_add3:   0%|          | 0/2672 [00:00<?, ?it/s]

deberta | delete_only_del5:   0%|          | 0/2672 [00:00<?, ?it/s]

deberta | hybrid_add3_delete5:   0%|          | 0/2672 [00:00<?, ?it/s]

,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,add_only_add3,2672,0.004491,0.004896,0.989521,0.996997,0.982249,0.989568,0.999674
1,delete_only_del5,2672,0.007111,0.007517,0.988398,0.998491,0.978550,0.988420,0.999657
2,hybrid_add3_delete5,2672,0.016841,0.016859,0.978668,0.998460,0.959320,0.978499,0.999619


In [27]:
val_evasion_summaries = []
val_evasion_details = {}

for attack_name, attack_fn in attack_specs:
    summary, details = evaluate_evasion_on_phishing(
        val_df,
        attack_name,
        attack_fn,
        show_progress=True
    )
    val_evasion_summaries.append(summary)
    val_evasion_details[attack_name] = details

val_evasion_results_df = pd.DataFrame(val_evasion_summaries).sort_values("attack_success_rate", ascending=False)
val_evasion_results_df

deberta | add_only_add3 phishing:   0%|          | 0/1339 [00:00<?, ?it/s]

deberta | delete_only_del5 phishing:   0%|          | 0/1339 [00:00<?, ?it/s]

deberta | hybrid_add3_delete5 phishing:   0%|          | 0/1339 [00:00<?, ?it/s]

,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
2,hybrid_add3_delete5,1352,1339,42,0.031367,0.968633,0.030794
1,delete_only_del5,1352,1339,16,0.011949,0.988051,0.012194
0,add_only_add3,1352,1339,11,0.008215,0.991785,0.008411


In [28]:
all_test_attack_results = {}
all_test_evasion_results = {}
attack_specs = [
    ("delete_only_del5", lambda x: delete_only_attack(x, delete_steps=5)),
    ("hybrid_add3_delete5", lambda x: hybrid_add_then_delete_attack_blackbox(x, add_steps=3, delete_steps=5)),
]

for test_df in test_dfs:
    dataset_name = test_df["dataset"].iloc[0]

    print(f"\n\n==============================")
    print(f"Starting dataset: {dataset_name}")
    print(f"Rows: {len(test_df)}")
    print(f"==============================")

    attack_summaries = []
    evasion_summaries = []

    for attack_name, attack_fn in attack_specs:
        print(f"\nRunning attack: {attack_name}")

        summary_attack, _ = evaluate_attack_detailed(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        attack_summaries.append({"dataset": dataset_name, **summary_attack})

        summary_evasion, _ = evaluate_evasion_on_phishing(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        evasion_summaries.append({"dataset": dataset_name, **summary_evasion})

        print("\nAttack metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_attack}]))

        print("\nPhishing evasion metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_evasion}]))

    dataset_attack_df = pd.DataFrame(attack_summaries)
    dataset_evasion_df = pd.DataFrame(evasion_summaries)

    all_test_attack_results[dataset_name] = dataset_attack_df
    all_test_evasion_results[dataset_name] = dataset_evasion_df

    print(f"\nFinished dataset: {dataset_name}")

    print("\nAll attack metrics for this dataset:")
    display(dataset_attack_df)

    print("\nAll phishing evasion metrics for this dataset:")
    display(dataset_evasion_df)



Starting dataset: CEAS_08_cleaned
Rows: 39154

Running attack: delete_only_del5


deberta | delete_only_del5:   0%|          | 0/39154 [00:00<?, ?it/s]

deberta | delete_only_del5 phishing:   0%|          | 0/6595 [00:00<?, ?it/s]


Attack metrics:


,dataset,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,CEAS_08_cleaned,delete_only_del5,39154,0.111585,0.10836,0.502503,0.975836,0.110933,0.199219,0.935335


...

,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
0,SpamAssasin_cleaned,hybrid_add3_delete5,1718,520,318,0.611538,0.388462,0.568346



Finished dataset: SpamAssasin_cleaned

All attack metrics for this dataset:


,dataset,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,SpamAssasin_cleaned,delete_only_del5,5809,0.042692,0.039653,0.754519,0.993243,0.171129,0.291956,0.944239
1,SpamAssasin_cleaned,hybrid_add3_delete5,5809,0.056292,0.054962,0.739198,0.990338,0.119325,0.212987,0.949503



All phishing evasion metrics for this dataset:


,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
0,SpamAssasin_cleaned,delete_only_del5,1718,520,234,0.450000,0.550000,0.405501
1,SpamAssasin_cleaned,hybrid_add3_delete5,1718,520,318,0.611538,0.388462,0.568346


In [26]:
all_test_attack_results = {}
all_test_evasion_results = {}
attack_specs = [
    ("add_only_add3", lambda x: add_only_attack(x, add_steps=3))
]

for test_df in test_dfs:
    dataset_name = test_df["dataset"].iloc[0]

    print(f"\n\n==============================")
    print(f"Starting dataset: {dataset_name}")
    print(f"Rows: {len(test_df)}")
    print(f"==============================")

    attack_summaries = []
    evasion_summaries = []

    for attack_name, attack_fn in attack_specs:
        print(f"\nRunning attack: {attack_name}")

        summary_attack, _ = evaluate_attack_detailed(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        attack_summaries.append({"dataset": dataset_name, **summary_attack})

        summary_evasion, _ = evaluate_evasion_on_phishing(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        evasion_summaries.append({"dataset": dataset_name, **summary_evasion})

        print("\nAttack metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_attack}]))

        print("\nPhishing evasion metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_evasion}]))

    dataset_attack_df = pd.DataFrame(attack_summaries)
    dataset_evasion_df = pd.DataFrame(evasion_summaries)

    all_test_attack_results[dataset_name] = dataset_attack_df
    all_test_evasion_results[dataset_name] = dataset_evasion_df

    print(f"\nFinished dataset: {dataset_name}")

    print("\nAll attack metrics for this dataset:")
    display(dataset_attack_df)

    print("\nAll phishing evasion metrics for this dataset:")
    display(dataset_evasion_df)



Starting dataset: CEAS_08_cleaned
Rows: 39154

Running attack: add_only_add3


deberta | add_only_add3:   0%|          | 0/39154 [00:00<?, ?it/s]

deberta | add_only_add3 phishing:   0%|          | 0/6750 [00:00<?, ?it/s]


Attack metrics:


,dataset,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,CEAS_08_cleaned,add_only_add3,39154,0.0792,0.067943,0.547505,0.982908,0.192199,0.321526,0.936929


...

,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
0,SpamAssasin_cleaned,add_only_add3,1718,530,165,0.311321,0.688679,0.283547



Finished dataset: SpamAssasin_cleaned

All attack metrics for this dataset:


,dataset,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,SpamAssasin_cleaned,add_only_add3,5809,0.029265,0.028894,0.766569,0.986559,0.21362,0.351196,0.942586



All phishing evasion metrics for this dataset:


,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
0,SpamAssasin_cleaned,add_only_add3,1718,530,165,0.311321,0.688679,0.283547


In [ ]:
combined_attack_df = pd.concat(all_test_attack_results.values(), ignore_index=True)
combined_evasion_df = pd.concat(all_test_evasion_results.values(), ignore_index=True)

print("Combined attack results:")
print(combined_attack_df)

print("\nCombined evasion results:")
print(combined_evasion_df)

os.makedirs("/content/results", exist_ok=True)

distilbert_results_df.to_csv("/content/results/distilbert_clean_results.csv", index=False)
val_attack_results_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_val_add_delete_hybrid_metrics.csv", index=False)
val_evasion_results_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_val_add_delete_hybrid_evasion.csv", index=False)
combined_attack_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_test_add_delete_hybrid_metrics.csv", index=False)
combined_evasion_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_test_add_delete_hybrid_evasion.csv", index=False)

print("Saved DistilBERT result files to /content/results")